# 12 — Quantization and Deployment: PTQ/QAT, Serving, Export

Goal: ship models efficiently and understand quantization workflows.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Dynamic quantization (simple CPU path)

Dynamic quantization is often the easiest win for Linear-heavy models.

In [ ]:

import torch
import torch.nn as nn

try:
    import torch.ao.quantization as tq
    print("torch.ao.quantization available")
except Exception as e:
    tq = None
    print("quantization not available:", e)

model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2)).eval()

if tq is not None:
    qdyn = tq.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
    x = torch.randn(4,10)
    y = qdyn(x)
    print("dynamic quant output:", y.shape)

## 2. Serving sketch (FastAPI)

Load once, inference_mode in handlers, consider batching.

In [ ]:

fastapi_template = r'''
import torch
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
model = ...  # load model weights
model.eval()

class Inp(BaseModel):
    x: list[float]

@app.post("/predict")
def predict(inp: Inp):
    with torch.inference_mode():
        x = torch.tensor(inp.x).float().unsqueeze(0)
        logits = model(x)
        probs = torch.softmax(logits, dim=-1).squeeze(0).tolist()
    return {"probs": probs}
'''
print(fastapi_template)

## 3. Export overview

- TorchScript (legacy)
- torch.export (modern capture)
- ONNX (interoperability)